In [ ]:
!pip install -q "langchain==0.2.16" "langchain-community==0.2.16" langchain-huggingface faiss-cpu pypdf transformers torch langgraph

In [ ]:
!pip install "numpy<2.0" --force-reinstall

In [ ]:
import os
import warnings
from typing import TypedDict, List
from langchain_community.document_loaders import PyPDFLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain.prompts import PromptTemplate
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langgraph.graph import StateGraph, END

warnings.filterwarnings("ignore")
import re, requests
import pandas as pd
from langchain_core.documents import Document

In [ ]:
# ==============================================================================
# Step 1: Define Global State & Initialize LLM
# ==============================================================================

# Define the state that will be passed between agents in our workflow
class AgentState(TypedDict):
    task: str
    vector_store: FAISS
    retrieved_data: str
    processed_summary: str
    final_output_path: str
    log: List[str]
    # chat support (optional; the agents use .get() with defaults)
    last_question: str   # previous user question (for follow-ups like "what is the price of it?")
    last_answer: str     # previous answer (for "summarize that")
    mode: str            # qa / recommend / rank / summarize_list / summarize / no_info / smalltalk / nothing / nothing_to_rank
    reco: dict           # what the symptom-understanding step found (symptoms, child, red flags, cheapest/priciest, ...)
    docs: list           # chunks kept after re-ranking
    context: str         # text given to the LLM
    # chat history about the LAST list of drugs shown to the user (so "summarize" / "get the cheapest" work)
    last_pool: list      # every drug candidate of the previous recommendation (what "cheapest" compares)
    last_shown: list     # the drugs that were actually displayed last time (what "summarize" shortens)
    pool: list           # candidates of this turn
    shown: list          # drugs displayed in this turn

def init_llm():
    print("Loading local HuggingFace LLM (TinyLlama)...")
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=200,
        do_sample=False,          # deterministic -> fewer hallucinations
        repetition_penalty=1.1,
        return_full_text=False    # return only the answer, not the prompt
    )
    return HuggingFacePipeline(pipeline=pipe)

# Initialize locally
local_llm = init_llm()

In [ ]:
# ==============================================================================
# Step 2: Local Data Integration
# ==============================================================================
def clean_inventory_csv(csv_path, out_path="inventory_clean.csv"):
    # The sheet has blank/title rows above the real header + an empty first column,
    # so CSVLoader would read garbage. This keeps only the real table.
    raw = pd.read_csv(csv_path, header=None, dtype=str, keep_default_na=False)
    hdr = raw.index[raw.apply(lambda r: "Drug ID" in r.values, axis=1)][0]
    df = raw.iloc[hdr + 1:].copy()
    df.columns = [c.strip() for c in raw.iloc[hdr]]
    df = df.loc[:, [c for c in df.columns if c]]
    df = df[df["Drug ID"].str.startswith("DRUG")]
    df.to_csv(out_path, index=False)
    return out_path

def split_pdf_by_drug(pdf_path):
    # The guide is a TABLE of 120 numbered drugs ("1. Panadol Advance", "2. Cetal Tablets", ...)
    # -> one chunk per drug (keeps name + ingredient + price + dosage + pediatric safety together)
    text = "\n".join(p.page_content for p in PyPDFLoader(pdf_path).load())
    # drop everything after the disclaimer, page footers, section titles and the repeated table header
    text = re.split(r"Important Medical Disclaimer", text)[0]
    text = re.sub(r"^.*Comprehensive Pharmacy AI Knowledge Base.*$", "", text, flags=re.M)
    text = re.sub(r"^Section \d+:.*$", "", text, flags=re.M)
    text = re.sub(r"^Trade & Active\s*\nIngredient\s*\nEst\.\s*\nPrice.*$", "", text, flags=re.M)
    # a new drug starts with "<1-3 digits>. <Capital letter>" at the beginning of a line
    parts = re.split(r"\n(?=\d{1,3}\.\s+[A-Z])", text)
    docs = []
    for p in parts:
        p = p.strip()
        if re.match(r"\d{1,3}\.\s", p):
            docs.append(Document(page_content=p, metadata={"source": pdf_path}))
    return docs

# Teach the "foreign drug" check about every drug that exists in the two files
GENERIC_WORDS = {"vitamin", "multivitamins", "acid", "sodium", "extract", "extracts", "calcium", "syrup",
                 "tablets", "capsules", "cream", "drops", "oil", "plus", "combo", "pediatric"}
def register_drug_names(pdf_docs, csv_path):
    names = set()
    for d in pdf_docs:                       # "17. Augmentin Syrup" -> augmentin
        m = re.match(r"\d{1,3}\.\s+([A-Za-z\-]+)", d.page_content)
        if m: names.add(m.group(1).lower())
    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    for col in ("Trade Name", "Active Ingredient"):
        for v in df[col]:
            for w in re.findall(r"[a-z]+", v.lower()):
                names.add(w)
    DRUG_TERMS.update(w for w in names if len(w) >= 4 and w not in GENERIC_WORDS)

def load_and_process_documents(pdf_path: str, csv_path: str):
    print("Loading local documents...")
    documents = []

    pdf_docs = []
    if os.path.exists(pdf_path):
        pdf_docs = split_pdf_by_drug(pdf_path)
        documents.extend(pdf_docs)
    else:
        print(f"Warning: PDF file not found at {pdf_path}")

    if os.path.exists(csv_path):
        clean_csv = clean_inventory_csv(csv_path)
        documents.extend(CSVLoader(clean_csv).load())
        register_drug_names(pdf_docs, clean_csv)
        build_recommender(pdf_docs, clean_csv)      # Step 3b: structured drug table for symptom -> drug recommendations
    else:
        print(f"Warning: CSV file not found at {csv_path}")

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=50)
    chunks = text_splitter.split_documents(documents)
    print(f"Loaded {len(documents)} documents -> {len(chunks)} chunks")

    print("Initializing HuggingFace Embeddings & FAISS...")
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(chunks, embeddings)
    return vector_store

In [ ]:
# ==============================================================================
# Step 3: Define AI Agents (Data Retrieval, Processing, Action)
# ==============================================================================

# Real external API: openFDA drug labels (free, no key). Looks up the drug named in the question / retrieved text.
FDA_NAMES = {"paracetamol": "acetaminophen", "ibuprofen": "ibuprofen", "diclofenac": "diclofenac",
             "cetirizine": "cetirizine", "omeprazole": "omeprazole", "azithromycin": "azithromycin",
             "amoxicillin": "amoxicillin", "pseudoephedrine": "pseudoephedrine", "metformin": "metformin",
             "atorvastatin": "atorvastatin", "loratadine": "loratadine", "salbutamol": "albuterol"}

def fetch_fda_label(context: str) -> str:
    low = context.lower()
    found = [(low.index(n), n) for n in FDA_NAMES if n in low]
    if not found:
        return "No drug name recognised for the FDA lookup."
    name = FDA_NAMES[min(found)[1]]
    try:
        r = requests.get("https://api.fda.gov/drug/label.json",
                         params={"search": f'openfda.generic_name:"{name}"', "limit": 1}, timeout=8)
        res = r.json()["results"][0]
        warn = (res.get("warnings") or res.get("boxed_warning") or [""])[0][:300]
        return f"openFDA label for {name}: {warn}"
    except Exception as e:
        return f"openFDA API unavailable ({type(e).__name__})."

# ------------------------------------------------------------------------------
# Helpers: fixed replies, query expansion, relevance gate, answer validation
# ------------------------------------------------------------------------------
NO_INFO = ("I don't have information about that. I can only answer questions about the "
           "drugs in the medical guide and the pharmacy inventory.")
GREETING_REPLY = "Hi! Ask me about a drug's dosage, warnings, price, or stock."
NOTHING_TO_SUMMARIZE = "There is nothing to summarize yet. Ask me a question about a drug first."
NOTHING_TO_RANK = ("There is no previous list to compare yet. Tell me what you need first "
                   "(for example: 'I have a cold') and then ask for the cheapest one.")
FIXED_REPLIES = {NO_INFO, GREETING_REPLY, NOTHING_TO_SUMMARIZE, NOTHING_TO_RANK}

# Generic words that say nothing about WHICH drug the question is about
STOP = {"what", "which", "with", "that", "this", "from", "does", "have", "about", "there", "their",
        "would", "could", "should", "please", "tell", "give", "many", "much", "long", "take", "name",
        "used", "label", "price", "cost", "stock", "dose", "dosage", "strength", "available",
        "availability", "manufacturer", "expiry", "storage", "shelf", "location", "indication",
        "indications", "warning", "warnings", "contraindication", "contraindications", "side",
        "effects", "maximum", "minimum", "daily", "drug", "drugs", "medicine", "medication",
        "tablet", "tablets", "capsule", "standard", "inventory"}

# Words too common in English to prove an answer is grounded in the context
COMMON = {"that", "this", "with", "from", "have", "which", "there", "their", "they", "when", "been",
          "will", "should", "would", "also", "than", "your", "into", "only", "according", "based",
          "context", "provided", "information", "about", "answer", "question", "these", "those"}

DRUG_TERMS = ({"panadol", "adol", "brufen", "advil", "cataflam", "fastum", "congestal", "zyrtec",
               "antinal", "omez", "prilosec", "buscopan", "augmentin", "zithromax"}
              | set(FDA_NAMES) | set(FDA_NAMES.values()))

GREETING = re.compile(r"^\s*(hi|hello|hey|thanks|thank you)[\s!.?]*$", re.I)
SUMMARIZE = re.compile(r"\b(summar\w*|shorten|shorter|tl;?dr|simplif\w*|simpler)\b", re.I)
FOLLOW_UP = re.compile(r"\b(that|it|this|them|those|again)\b", re.I)

def keywords(text):
    # Distinctive words of a text (plural 's' removed) - mainly drug names / conditions
    out = set()
    for w in re.findall(r"[a-z0-9]+", text.lower()):
        if len(w) >= 4 and w not in STOP:
            out.add(w[:-1] if w.endswith("s") and len(w) > 4 else w)
    return out

def expand_query(task, last_question):
    # Follow-up such as "what is the price of it?" -> add the previous question so the search knows the drug
    if last_question and FOLLOW_UP.search(task):
        return f"{last_question} {task}"
    return task

def rerank_docs(task, docs, top_n=2):
    # Keep the chunks that contain the distinctive words of the question (e.g. the drug name).
    # Also returns `matched`: False means nothing retrieved is really about the question.
    words = keywords(task)
    texts = [d.page_content.lower() for d in docs]
    useful = [w for w in words if sum(w in t for t in texts) <= len(docs) / 2]
    score = lambda i: sum(w in texts[i] for w in useful)
    order = sorted(range(len(docs)), key=lambda i: (-score(i), i))[:top_n]
    return [docs[i] for i in order], any(score(i) > 0 for i in order)

def retrieve_docs(vector_store, query, k=8, top_n=2):
    return rerank_docs(query, vector_store.similarity_search(query, k=k), top_n=top_n)

def ask_llm(system, user):
    # TinyLlama chat format
    prompt = f"<|system|>\n{system}</s>\n<|user|>\n{user}</s>\n<|assistant|>\n"
    out = str(local_llm.invoke(prompt))
    if "<|" in out:
        out = out.split("<|")[0]
    return out.strip()

def _stems(text):
    return {t[:5] for t in re.findall(r"[a-z0-9]+", text.lower()) if len(t) >= 4 and t not in COMMON}

def is_grounded(answer, context, min_ratio=0.6):
    # Is the answer usable? (not empty, not "I don't know", no foreign drug, words come from the context)
    if len(answer.strip()) < 3:
        return False
    low = answer.lower()
    if re.search(r"i don'?t know|i do not know|not (?:provided|mentioned)|cannot answer", low):
        return False
    if (set(re.findall(r"[a-z]+", low)) & DRUG_TERMS) - set(re.findall(r"[a-z]+", context.lower())):
        return False
    a, c = _stems(answer), _stems(context)
    return not a or len(a & c) / len(a) >= min_ratio

def extractive_answer(docs, max_chars=700):
    # Safe fallback: show the retrieved facts directly (no LLM involved)
    parts = []
    for d in docs[:2]:
        lines = [l.strip() for l in d.page_content.splitlines() if l.strip()]
        is_pdf = str(d.metadata.get("source", "")).lower().endswith(".pdf")
        parts.append((" " if is_pdf else "; ").join(lines))
    text = "Here is what I found:\n\n" + "\n\n".join(parts)
    return text if len(text) <= max_chars else text[:max_chars].rsplit(" ", 1)[0] + "..."

def summarize_text(text):
    # Up to 2 LLM tries, then a deterministic fallback (first two sentences)
    for system in ("Summarize the text in 1-2 short sentences. Use only facts from the text.",
                   "Rewrite the text as one short sentence. Use only the words and facts in the text."):
        out = ask_llm(system, text)
        if is_grounded(out, text, 0.7) and len(out) < len(text):
            return out
    return " ".join(re.split(r"(?<=[.!?])\s+", text.strip())[:2])

QA_SYSTEM = ("You are a pharmacy assistant. Answer the question using ONLY the context. "
             "Talk only about the drug in the question. Never mention other drugs. "
             "If the answer is not in the context, say: I don't know. Keep the answer to 1-3 sentences.")

def answer_question(state):
    task = state["task"]

    # Try 1: the best 2 chunks
    context = state["context"]
    out = ask_llm(QA_SYSTEM, f"Context:\n{context}\n\nQuestion: {task}")
    if is_grounded(out, context):
        state["log"].append("Processing: answered on try 1.")
        return out

    # Try 2: expand the search (more candidates, 3 chunks) and ask again
    query = expand_query(task, state.get("last_question", ""))
    docs, _ = retrieve_docs(state["vector_store"], query, k=12, top_n=3)
    context2 = "\n\n".join(d.page_content for d in docs)
    if "fda" in task.lower():
        context2 += "\n\n" + state["retrieved_data"].split("EXTERNAL API CONTEXT:\n")[-1]
    out = ask_llm(QA_SYSTEM + " Answer briefly.", f"Context:\n{context2}\n\nQuestion: {task}")
    if is_grounded(out, context2):
        state["log"].append("Processing: answered on try 2 (expanded search).")
        return out

    # Fallback: show the retrieved facts directly
    state["log"].append("Processing: LLM answer rejected twice, used extractive fallback.")
    return extractive_answer(state["docs"] or docs)

# ------------------------------------------------------------------------------
# Agent 1: Data Retrieval Agent
# Responsibility: decides what the message is (question / symptom / follow-up on the previous list /
# summary / small talk), fetches data from the vector DB (PDF + Sheet) and, if asked, the openFDA API.
# It reads the chat history (last_pool / last_shown) so that:
#   "summarize"        -> shortens the previous list to 2 drugs
#   "get the cheapest" -> picks the cheapest drug of the previous list
# ------------------------------------------------------------------------------
def data_retrieval_agent(state: AgentState):
    print("-> [Agent 1: Retrieval] Gathering data from VectorDB and External APIs...")
    task = state["task"]
    last_q = state.get("last_question", "")
    last_a = state.get("last_answer", "")
    last_pool = state.get("last_pool") or []
    last_shown = state.get("last_shown") or []
    state.update({"mode": "qa", "docs": [], "context": "", "pool": [], "shown": []})

    try:
        info = understand(task)                      # Step 3b: symptoms / child / red flags / cheapest-priciest
        state["reco"] = info
        rank = info["rank"]
        if GREETING.match(task):
            state["mode"] = "smalltalk"
        elif SUMMARIZE.search(task) and not info["drug_named"]:
            if last_shown:                           # the last answer was a list of drugs -> keep only 2
                state["mode"] = "summarize_list"
                state["shown"], state["pool"] = last_shown, last_pool
                state["retrieved_data"] = "PREVIOUS LIST:\n" + items_context(last_shown)
            else:                                    # the last answer was plain text -> summarize the text
                has_answer = bool(last_a.strip()) and last_a not in FIXED_REPLIES
                state["mode"] = "summarize" if has_answer else "nothing"
                state["context"] = last_a
                state["retrieved_data"] = f"PREVIOUS ANSWER:\n{last_a}"
        elif info["self_harm"] or info["red_flag"]:
            state["mode"] = "recommend"              # safety message, no drug is suggested
            state["retrieved_data"] = ""
        elif rank and not info["symptoms"] and not info["drug_named"]:
            # "get the cheapest" / "and the most expensive?" -> use the previous list (chat history)
            if last_pool:
                state["mode"] = "rank"
                state["pool"] = last_pool
                state["shown"] = [pick_by_price(last_pool, rank)]
                state["retrieved_data"] = "PREVIOUS LIST:\n" + items_context(last_pool)
            else:
                state["mode"] = "nothing_to_rank"
        elif info["symptoms"] and not info["drug_named"]:
            # "I have a cold, recommend something" (or "cheapest medicine for a headache")
            pool = build_items(info, top_n=6 if rank else 3)
            state["mode"] = "recommend"
            state["pool"] = pool
            state["shown"] = [pick_by_price(pool, rank)] if (rank and pool) else pool
            state["retrieved_data"] = "RECOMMENDATION CONTEXT:\n" + items_context(pool)
            state["context"] = state["retrieved_data"]
        else:
            query = expand_query(task, last_q)
            docs, matched = retrieve_docs(state["vector_store"], query)
            local_context = "\n\n".join(d.page_content for d in docs)

            external = "Not requested."
            wants_fda = "fda" in task.lower()
            if wants_fda:
                external = fetch_fda_label(task + "\n" + local_context)
                if any(n in task.lower() for n in FDA_NAMES):
                    matched = True

            state["docs"] = docs
            state["context"] = local_context + (f"\n\n{external}" if wants_fda else "")
            state["retrieved_data"] = f"LOCAL CONTEXT:\n{local_context}\n\nEXTERNAL API CONTEXT:\n{external}"
            if not matched:
                state["mode"] = "no_info"   # nothing in the data is about this question
    except Exception as e:
        state["mode"] = "no_info"
        state["retrieved_data"] = f"Retrieval error: {type(e).__name__}: {e}"

    state["log"].append(f"Data Retrieval Agent completed (mode: {state['mode']}).")
    return state

# ------------------------------------------------------------------------------
# Agent 2: Processing & Summarization Agent
# Responsibility: answers / summarizes with the LLM, validates the result and retries.
# Lists of drugs (recommend / rank / summarize_list) use fixed templates -> no hallucinated drugs.
# ------------------------------------------------------------------------------
def processing_agent(state: AgentState):
    print("-> [Agent 2: Processor] Analyzing and summarizing information...")
    mode = state.get("mode", "qa")
    try:
        if mode == "smalltalk":
            summary = GREETING_REPLY
        elif mode == "recommend":
            summary = format_recommendation(state["reco"], state["shown"], state["pool"])
        elif mode == "rank":
            summary = format_rank(state["reco"], state["shown"], state["pool"], from_history=True)
        elif mode == "summarize_list":
            summary = format_two_drug_summary(state["shown"])
        elif mode == "nothing":
            summary = NOTHING_TO_SUMMARIZE
        elif mode == "nothing_to_rank":
            summary = NOTHING_TO_RANK
        elif mode == "no_info":
            summary = NO_INFO
        elif mode == "summarize":
            summary = summarize_text(state["context"])
        else:
            summary = answer_question(state)
    except Exception as e:
        print(f"   Processor error: {type(e).__name__}: {e}")
        summary = extractive_answer(state["docs"]) if state.get("docs") else NO_INFO

    state["processed_summary"] = summary
    state["log"].append("Processing Agent completed.")
    return state

# ------------------------------------------------------------------------------
# Agent 3: Automated Action Agent
# Responsibility: executes an automated output (saves a markdown report).
# ------------------------------------------------------------------------------
def action_agent(state: AgentState):
    print("-> [Agent 3: Action] Executing automated action (Generating Report)...")
    summary = state["processed_summary"]
    output_path = "automated_report.md"
    try:
        report_content = f"# Automated System Report\n\n## Processed Output\n{summary}\n\n## Status\nGenerated successfully by AI Orchestration Pipeline."
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(report_content)
        state["log"].append(f"Action Agent completed. Report saved to {output_path}.")
    except Exception as e:
        output_path = ""
        state["log"].append(f"Action Agent failed: {type(e).__name__}.")

    state["final_output_path"] = output_path
    return state


In [ ]:
# ==============================================================================
# Step 3b: Understanding the user's own words (symptom -> drug recommendation)
# English only. It reads ONLY the two data files (medical guide PDF + inventory CSV).
# It also keeps the last list of drugs, so follow-ups like "summarize" or
# "get the cheapest" work on what the user just saw.
# ==============================================================================

# ---------- 1) text helpers ----------
def normalize(text):
    return text.lower()

def W(*words):
    # One regex from English words/phrases; every word must start at a word boundary.
    # (words may contain regex such as "flu\b" so that "flu" does not match "fluid")
    return re.compile("|".join(r"\b" + w for w in words))

# ---------- 2) what the user may complain about ----------
# pats  : words/phrases the user may say
# terms : (regex, weight) searched in the guide's "Primary Indications" text and in the inventory's "Primary Indication"
# rx    : True = these are prescription medicines -> tell the user to see a doctor
SYMPTOMS = {
 "cold_flu":   dict(en="cold / flu",
                    pats=W("cold", r"flu\b", "influenza", "runny nose", "sniffl"),
                    terms=[(r"\bcold\b|\bflu\b", 3), (r"congestion|rhinitis|sinus", 1)]),
 "fever":      dict(en="fever",
                    pats=W("fever", "temperature", "high temp"),
                    terms=[(r"fever|antipyretic", 3)]),
 "headache":   dict(en="headache",
                    pats=W("headache", "head ache"),
                    terms=[(r"headache", 3), (r"migraine", 1)]),
 "migraine":   dict(en="migraine",
                    pats=W("migraine"),
                    terms=[(r"migraine", 4)]),
 "pain":       dict(en="pain",
                    pats=W("pain", "ache", "hurts"),
                    terms=[(r"pain", 2), (r"analgesic|relief", 1)]),
 "toothache":  dict(en="toothache",
                    pats=W("toothache", "tooth", "dental", "teeth"),
                    terms=[(r"dental|toothache", 4)]),
 "muscle_joint": dict(en="muscle / joint pain",
                    pats=W("joint", "muscle", "sprain", "back pain", "arthritis", "rheumat"),
                    terms=[(r"\bjoint|(?<!smooth )muscle|sprain|arthritis|rheumat|osteoarthritis|back pain|contusion", 3)]),
 "dysmenorrhea": dict(en="period pain",
                    pats=W("period pain", "period cramps", "menstrual", "dysmenorrhea"),
                    terms=[(r"dysmenorrhea|menstrual", 4)]),
 "sore_throat": dict(en="sore throat",
                    pats=W("sore throat", "throat", "tonsil"),
                    terms=[(r"sore throat", 5), (r"throat|pharyn", 2), (r"tonsil", 1)]),
 "cough_dry":  dict(en="dry cough",
                    pats=W("dry cough"),
                    terms=[(r"dry irritating cough|suppressant", 5), (r"cough", 1)]),
 "cough_wet":  dict(en="chesty cough / phlegm",
                    pats=W("phlegm", "mucus", "productive cough", "wet cough", "chesty"),
                    terms=[(r"mucus|mucolytic|expectorant|productive cough|bronchitis", 3), (r"cough", 1)]),
 "cough":      dict(en="cough",
                    pats=W("cough"),
                    terms=[(r"cough", 3), (r"mucus|bronchitis", 1)]),
 "allergy":    dict(en="allergy",
                    pats=W("allerg", "hives", "sneez", "itch", "ur?ticaria"),
                    terms=[(r"(?<!penicillin-)allerg|urticaria|sneezing|itching", 4)]),
 "stuffy_nose": dict(en="blocked nose",
                    pats=W("blocked nose", "stuffy", "nasal", "congest", "nose is blocked"),
                    terms=[(r"nasal|decongest|congestion", 3)]),
 "sinus":      dict(en="sinusitis",
                    pats=W("sinus"),
                    terms=[(r"sinus", 4)]),
 "asthma":     dict(en="asthma / wheezing / shortness of breath",
                    pats=W("asthma", "wheez", "breath"),
                    terms=[(r"asthma|wheez|bronchospasm|shortness of breath", 4)]),
 "diarrhea":   dict(en="diarrhea",
                    pats=W("diarrh", "loose stool"),
                    terms=[(r"diarrh", 4)]),
 "constipation": dict(en="constipation",
                    pats=W("constipat"),
                    terms=[(r"constipation|laxative", 4)]),
 "heartburn":  dict(en="heartburn / acidity / reflux",
                    pats=W("heartburn", "acid", "reflux", "gerd", "indigestion"),
                    terms=[(r"heartburn|reflux|gerd|hyperacidity|peptic|acid", 3)]),
 "gas_bloating": dict(en="gas / bloating",
                    pats=W(r"gas\b", "bloat", "flatul"),
                    terms=[(r"\bgas\b|bloating|flatul|antiflatulent", 4)]),
 "stomach_cramps": dict(en="stomach cramps / colic",
                    pats=W("stomach ache", "stomach pain", "cramp", "colic", "abdominal pain", "tummy"),
                    terms=[(r"colic|cramp|(?<!broncho)spasm|antispasmodic", 4)]),
 "nausea":     dict(en="nausea / vomiting",
                    pats=W("nausea", "vomit", "throwing up"),
                    terms=[(r"nausea|vomiting", 4)]),
 "vertigo":    dict(en="dizziness / vertigo / motion sickness",
                    pats=W("dizz", "vertigo", "motion sickness"),
                    terms=[(r"vertigo|motion sickness|tinnitus", 4)]),
 "insomnia":   dict(en="insomnia",
                    pats=W("insomnia", "can'?t sleep", "sleepless"),
                    terms=[(r"insomnia", 5)], rx=True),
 "anxiety":    dict(en="anxiety / depression",
                    pats=W("anxiety", "anxious", "depress", "panic"),
                    terms=[(r"anxiety|depress|panic", 4)], rx=True),
 "nerve_pain": dict(en="nerve pain / neuritis",
                    pats=W("neuropath", "neuritis", "nerve"),
                    terms=[(r"neuropath|neuritis|neuralgia|nerve", 4)]),
 "hypertension": dict(en="high blood pressure",
                    pats=W("blood pressure", "hypertension"),
                    terms=[(r"hypertension|blood pressure|antihypertensive", 4)], rx=True),
 "diabetes":   dict(en="diabetes",
                    pats=W("diabet", "blood sugar"),
                    terms=[(r"diabet|t2dm|blood sugar|insulin", 4)], rx=True),
 "cholesterol": dict(en="high cholesterol",
                    pats=W("cholesterol", "lipid"),
                    terms=[(r"cholesterol|lipid|hyperlipidemia|\bldl\b", 4)], rx=True),
 "thyroid":    dict(en="thyroid",
                    pats=W("thyroid"),
                    terms=[(r"thyroid", 4)], rx=True),
 "epilepsy":   dict(en="epilepsy / seizures",
                    pats=W("epilep", "seizure", "convuls"),
                    terms=[(r"epilep|seizure|anticonvuls", 4)], rx=True),
 "infection":  dict(en="bacterial infection",
                    pats=W("infection", "antibiotic"),
                    terms=[(r"infection|antibiotic|bacterial|sinusitis|otitis", 3)], rx=True),
 "uti":        dict(en="urinary infection",
                    pats=W(r"uti\b", "urinary", "bladder", "burning urination"),
                    terms=[(r"\buti\b|urinary|bladder", 4)], rx=True),
 "worms_diarrhea_parasite": dict(en="amoebiasis / giardia",
                    pats=W("amoeb", "giardia", "parasite"),
                    terms=[(r"amoeb|giardia|trichomon", 4)], rx=True),
 "burn":       dict(en="burns / wounds",
                    pats=W("burn", "scald", "wound"),
                    terms=[(r"\bburn", 5), (r"wound", 2)]),
 "acne":       dict(en="acne",
                    pats=W("acne", "pimple"),
                    terms=[(r"acne", 5), (r"hyperpigment", 1)]),
 "fungal":     dict(en="fungal infection / thrush",
                    pats=W("fung", "thrush", "candid", "ringworm"),
                    terms=[(r"fung|thrush|candid", 4)]),
 "eczema":     dict(en="eczema / dermatitis / psoriasis",
                    pats=W("eczema", "dermatitis", "psoria"),
                    terms=[(r"eczema|dermatitis|psoria|dermatos", 4)]),
 "diaper_rash": dict(en="diaper rash / dry skin",
                    pats=W("diaper", "nappy", "dry skin", "chapped"),
                    terms=[(r"diaper|dry skin|chapped|skin barrier", 4)]),
 "scar":       dict(en="scars",
                    pats=W(r"scars?\b"),
                    terms=[(r"scar", 5)]),
 "dry_eye":    dict(en="dry / tired eyes",
                    pats=W("dry eye", "tired eye", "eye strain", r"eyes?\b"),
                    terms=[(r"eye drop|dry eye|ocular|lubricating|conjunctivitis", 4)]),
 "conjunctivitis": dict(en="red / allergic eyes",
                    pats=W("conjunctivitis", "red eye", "pink eye"),
                    terms=[(r"conjunctivitis|ocular|redness", 4)]),
 "ear":        dict(en="ear pain / earwax",
                    pats=W("earache", "ear pain", "earwax", "ear wax", r"ears?\b"),
                    terms=[(r"otitis|earwax|cerumen|ear drop", 4), (r"\bear\b", 2)]),
 "anemia":     dict(en="anemia / iron deficiency",
                    pats=W("anemi", "anaemi", r"iron\b"),
                    terms=[(r"anemia|iron", 4)]),
 "bones":      dict(en="bones / calcium / vitamin D",
                    pats=W("bone", "calcium", "osteopor", "vitamin d", "rickets"),
                    terms=[(r"bone|osteopor|calcium|rickets|vitamin d", 4)]),
 "immunity":   dict(en="immunity",
                    pats=W("immun", "vitamin c"),
                    terms=[(r"immun", 4)]),
 "hair_loss":  dict(en="hair loss",
                    pats=W("hair loss", "hair fall"),
                    terms=[(r"hair", 5)]),
 "energy":     dict(en="tiredness / low energy",
                    pats=W("tired", "fatigue", "low energy", "exhaust"),
                    terms=[(r"energy|stamina|vitality|fatigue", 4), (r"multivitamin", 1)]),
 "breast_milk": dict(en="low breast milk",
                    pats=W("breast milk", "lactation", "breastfeed"),
                    terms=[(r"breast milk|galactagogue|lactat", 5)]),
 "teething_colic_baby": dict(en="baby colic / teething",
                    pats=W("teething", "infant colic", "baby colic", "gripe"),
                    terms=[(r"teething", 4), (r"infant", 2)]),
 "contraception": dict(en="contraception / hormones",
                    pats=W("contracept", "birth control", "menopause"),
                    terms=[(r"contracept|hormone replacement|menstrual cycle", 4)], rx=True),
}

# Words that mean "go to a doctor / hospital now" -> we do NOT recommend a drug first
RED_FLAGS = W("chest pain", r"can'?t breathe", "cannot breathe", "difficulty breathing", "blood in", "vomiting blood",
              "faint", "unconscious", "seizure now", "swollen face", "swollen tongue", "overdose", "poison")

SELF_HARM = W("suicid", "kill myself", "end my life", "want to die", "hurt myself")

CHILD_WORDS = W("child", r"kids?\b", "baby", "infant", "toddler", "newborn", "my son", "my daughter",
                "pediatric", "paediatric", "teething", "diaper")

# Price words used in follow-ups such as "get the cheapest" / "and the most expensive one?"
CHEAPEST = re.compile(r"\b(cheap\w*|lowest[ -](?:price|cost)|least expensive|most affordable|inexpensive)\b", re.I)
PRICIEST = re.compile(r"\b(most expensive|priciest|highest[ -](?:price|cost)|costliest)\b", re.I)

# ---------- 3) Reading the two data files into structured records ----------
GUIDE = []        # one dict per drug of the PDF guide
INVENTORY = []    # one dict per drug of the CSV inventory
BRAND_WORDS = set()   # names/ingredients that show the user is asking about a specific drug

NAME_TAIL = re.compile(r"(?:Chew\s+)?(?:Suppositories|Elixir|Syrup|Flu Day|Tabs|Caps|Drops|Gel|Cream|Ointment|Liquid|Sachets|Lozenges|Caplets|Wash|Capsules|Tablets|Inhalant|Suspension)", re.I)

def parse_guide_chunk(chunk):
    # "17. Augmentin Syrup\n457\nAmoxicillin + Clavulanate\n85 EGP <indications>\n<dosage>\n<pediatric>" -> dict
    lines = [l.strip() for l in chunk.splitlines() if l.strip()]
    pi = next((i for i, l in enumerate(lines) if re.search(r"\d+(?:\.\d+)?\s*EGP", l)), None)
    if pi is None:
        return None
    head, m = lines[:pi], re.search(r"(\d+(?:\.\d+)?)\s*EGP\s*(.*)", lines[pi])
    name, k = re.sub(r"^\d{1,3}\.\s*", "", head[0]), 1
    while k < len(head) and (re.fullmatch(r"\d+", head[k]) or NAME_TAIL.fullmatch(head[k]) or name.endswith(("+", "-"))):
        name, k = (name + head[k] if name.endswith("-") else name + " " + head[k]), k + 1      # continuation of the product name
    rest = re.sub(r"\s+\.", ".", " ".join([m.group(2)] + lines[pi + 1:]))
    segs = [s.strip() for s in re.split(r"(?<!e\.g)\.(?=\s|$)", rest) if s.strip()]
    if len(segs) != 3:                      # unexpected layout -> keep everything in 'indication'
        segs = [rest, "", ""]
    return {"name": name, "ingredient": " ".join(head[k:]), "price": m.group(1),
            "indication": segs[0], "dose": segs[1], "pediatric": segs[2], "text": chunk}

def child_ok(ped):
    # Is the pediatric note of the guide good news for a child?  (kept strict on purpose)
    p = ped.lower()
    if re.search(r"contraindicated|adults?\b|not (?:for|suitable|recommended)|under \d|age \d+\+|above 1\d|strict|caution|supervision|monitor|per md|per pediatrician|specialist|short-term|very short", p):
        return False
    return bool(re.search(r"safe|essential|extremely|syrup|drops|ages? \d{1,2} to \d{1,2}|above \d\b|suitable above \d\b", p))

def build_recommender(pdf_docs, csv_path):
    GUIDE.clear(); INVENTORY.clear(); BRAND_WORDS.clear()
    for d in pdf_docs:
        r = parse_guide_chunk(d.page_content)
        if r: GUIDE.append(r)
    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    INVENTORY.extend(df.to_dict("records"))
    # which words in a user message mean "a specific drug" (brand names + long ingredient names)
    for r in GUIDE:
        m = re.match(r"[A-Za-z]{4,}", r["name"])
        if m: BRAND_WORDS.add(m.group(0).lower())
    for r in INVENTORY:
        m = re.match(r"[A-Za-z]{4,}", r["Trade Name"].strip())
        if m: BRAND_WORDS.add(m.group(0).lower())
    for src in [r["ingredient"] for r in GUIDE] + [r["Active Ingredient"] for r in INVENTORY]:
        for w in re.findall(r"[A-Za-z]{7,}", src.lower()):
            if w not in GENERIC_WORDS and w not in {"multivitamins", "extracts", "pediatric", "solution", "chloride", "fumarate", "potassium", "methylsulfate", "dihydrochloride", "propionate", "hydrochloride", "bromide", "sulfate", "enzymes", "minerals", "complex", "digestive", "antispasmodic", "botanical", "peroxide"}:
                BRAND_WORDS.add(w)
    BRAND_WORDS.update({"paracetamol", "ibuprofen", "diclofenac"})
    print(f"Recommender ready: {len(GUIDE)} guide drugs, {len(INVENTORY)} inventory drugs, {len(SYMPTOMS)} symptoms understood.")

# ---------- 4) Understanding the message ----------
def mentions_specific_drug(norm_text):
    return bool(set(re.findall(r"[a-z]{4,}", norm_text)) & BRAND_WORDS)

def understand(task):
    # -> dict(symptoms, child, red_flag, self_harm, drug_named, rank)
    norm = normalize(task)
    found = [key for key, s in SYMPTOMS.items() if s["pats"].search(norm)]
    # "dry cough"/"chesty cough" are more specific than plain "cough"
    if "cough" in found and ("cough_dry" in found or "cough_wet" in found):
        found.remove("cough")
    if "migraine" in found and "headache" in found:
        found.remove("headache")
    if "sinus" in found and "stuffy_nose" in found:
        found.remove("stuffy_nose")
    if "pain" in found and set(found) & {"stomach_cramps", "toothache", "muscle_joint", "ear", "sore_throat", "headache", "migraine", "dysmenorrhea", "nerve_pain", "dry_eye"}:
        found.remove("pain")                           # a more specific pain was named -> drop the generic one
    if "uti" in found and "infection" in found:
        found.remove("infection")
    rank = "priciest" if PRICIEST.search(task) else ("cheapest" if CHEAPEST.search(task) else None)
    return {"symptoms": found, "child": bool(CHILD_WORDS.search(norm)),
            "red_flag": bool(RED_FLAGS.search(norm)), "self_harm": bool(SELF_HARM.search(norm)),
            "drug_named": mentions_specific_drug(norm), "rank": rank}

# ---------- 5) Choosing drugs ----------
RX_INGREDIENTS = re.compile(r"amoxicillin|clavulan|azithromycin|ciproflox|levoflox|cefotax|ceftriax|clindamycin|metronidazole|doxycycline|erythromycin|sulfamethoxazole|flucloxacillin|ketorolac|bisoprolol|captopril|amlodipine|indapamide|sitagliptin|metformin|glimepiride|atorvastatin|rosuvastatin|levothyroxine|clopidogrel|warfarin|spironolactone|carbamazepine|valproate|pregabalin|zolmitriptan|flupentixol|estradiol|desogestrel|tobramycin|betamethasone|valsartan|nifuroxazide.*zzz", re.I)

def _score(text, terms):
    t, s = text.lower(), 0
    for rx, w in terms:
        if re.search(rx, t): s += w
    return s

def _first(name):
    m = re.search(r"[A-Za-z]+", name)
    return m.group(0).lower() if m else ""

def _stock_info(guide_name):
    first = _first(guide_name)
    for r in INVENTORY:
        if _first(r["Trade Name"]) == first:
            return r
    return None

def recommend(info, top_n=3):
    keys, child = info["symptoms"], info["child"]
    guide_hits, inv_hits = [], []
    for g in GUIDE:
        sc = sum(_score(g["indication"] + " " + g["name"], SYMPTOMS[k]["terms"]) for k in keys)
        if sc <= 0: continue
        ok = child_ok(g["pediatric"])
        if child and not ok: continue                       # never suggest a non-pediatric drug for a child
        inv = _stock_info(g["name"])
        rx = (inv["Prescription Required"].strip().lower() == "yes") if inv else bool(RX_INGREDIENTS.search(g["ingredient"]))
        in_stock = 1 if (inv and inv["Availability Status"] != "Out of Stock") else 0
        adj = sc + (1 if ok else 0)
        if not child and re.search(r"syrup|drops|suspension|water|suppositor", g["name"].lower()):
            adj -= 1.5                                       # adult asking -> kids' forms come after the adult forms
        guide_hits.append((adj, -int(rx), in_stock, -len(guide_hits), g, inv))
    guide_hits.sort(key=lambda x: x[:4], reverse=True)
    if not any(SYMPTOMS[k].get("rx") for k in keys):          # OTC complaint -> drop prescription drugs if there are enough OTC ones
        otc = [h for h in guide_hits if h[1] == 0]
        if len(otc) >= 2: guide_hits = otc
    for r in INVENTORY:
        sc = sum(_score(r["Primary Indication"], SYMPTOMS[k]["terms"]) for k in keys)
        if sc <= 0: continue
        if child and not re.search(r"child|pediatric|kid|children|baby", (r["Primary Indication"] + r["Trade Name"]).lower()):
            # inventory has no pediatric column -> for a child, accept only drugs that the guide marks as child-safe
            gg = next((g for g in GUIDE if _first(g["name"]) == _first(r["Trade Name"])), None)
            if not (gg and child_ok(gg["pediatric"])): continue
        avail = 0 if r["Availability Status"] == "Out of Stock" else 1
        inv_hits.append((sc, -int(r["Prescription Required"].strip().lower() == "yes"), avail, -len(inv_hits), r))
    inv_hits.sort(key=lambda x: x[:4], reverse=True)
    if not any(SYMPTOMS[k].get("rx") for k in keys):
        otc = [h for h in inv_hits if h[1] == 0]
        if len(otc) >= 1: inv_hits = otc
    return guide_hits[:top_n], inv_hits[:top_n]

# ---------- 6) Turning the matches into "items" (also what the chat remembers) ----------
def _price(v):
    m = re.search(r"\d+(?:\.\d+)?", str(v))
    return float(m.group(0)) if m else 0.0

def _is_rx(inv):
    return inv["Prescription Required"].strip().lower() == "yes"

def _stock_text(inv):
    return f"{inv['Availability Status']} ({inv['Stock Quantity']} units) - {inv['Location / Shelf']}"

def build_items(info, top_n=3):
    # one plain dict per recommended drug (guide drugs first, then inventory-only drugs)
    guide_hits, inv_hits = recommend(info, top_n)
    items, seen = [], set()
    for *_, g, inv in guide_hits:
        seen.add(_first(g["name"]))
        items.append({"source": "guide", "name": g["name"], "ingredient": g["ingredient"], "strength": "",
                      "price": _price(g["price"]), "dose": g["dose"], "pediatric": g["pediatric"],
                      "stock": _stock_text(inv) if inv else "",
                      "rx": _is_rx(inv) if inv else bool(RX_INGREDIENTS.search(g["ingredient"]))})
    for *_, r in inv_hits:
        if _first(r["Trade Name"]) in seen:        # already listed (with its stock) from the guide
            continue
        items.append({"source": "inventory", "name": r["Trade Name"], "ingredient": r["Active Ingredient"],
                      "strength": r["Strength"], "price": _price(r["Price (EGP)"]), "dose": "", "pediatric": "",
                      "stock": _stock_text(r), "rx": _is_rx(r)})
    return items

def _available(items):
    ok = [x for x in items if "out of stock" not in x["stock"].lower()]
    return ok or items

def pick_by_price(items, rank):
    # the cheapest / most expensive drug of a list (out-of-stock drugs are skipped when others exist)
    pool = _available(items)
    return min(pool, key=lambda x: x["price"]) if rank == "cheapest" else max(pool, key=lambda x: x["price"])

def items_context(items):
    # plain-text version of a list (used as 'retrieved_data' for the report/tests)
    return "\n".join(f"{x['name']} | {x['ingredient']} {x['strength']} | {x['price']:g} EGP | {x['dose']} | {x['pediatric']} | {x['stock']}"
                     for x in items)

# ---------- 7) Writing the answer (fixed templates, no LLM -> no hallucination) ----------
def _symptom_names(keys):
    return " + ".join(SYMPTOMS[k]["en"] for k in keys)

def _fmt_item(i, it, details=True):
    line = f"{i}. {it['name']} ({it['ingredient']})" + (f" {it['strength']}" if it["strength"] else "") + f" - {it['price']:g} EGP"
    if details and it["dose"]:
        line += f" | Dosage: {it['dose']}"
    if details and it["pediatric"]:
        line += f" | Children: {it['pediatric']}"
    if it["stock"]:
        line += f" | Inventory: {it['stock']}"
    if it["rx"]:
        line += " | Prescription required"
    return line

def format_two_drug_summary(shown):
    # "summarize" after a list -> only the top 2 drugs, one short line each
    L = [f"Summary - top {min(2, len(shown))} option{'s' if len(shown) > 1 else ''}:"]
    L += [_fmt_item(i, it, details=False) for i, it in enumerate(shown[:2], 1)]
    return "\n".join(L)

def format_rank(info, shown, pool, from_history):
    word = "cheapest" if info["rank"] == "cheapest" else "most expensive"
    where = "from the list above" if from_history else f"for ({_symptom_names(info['symptoms'])})" + (" in a child" if info["child"] else "")
    pick = shown[0]
    L = [f"The {word} option {where} is:", _fmt_item(1, pick)]
    if len(pool) > 1:
        skipped = len(pool) - len(_available(pool))
        L.append(f"(I compared {len(pool)} options" + (f"; {skipped} out-of-stock option(s) were skipped" if skipped else "") + ".)")
    if pick["rx"]:
        L.append("Note: this is a prescription medicine - a doctor must decide the right treatment.")
    if info["child"]:
        L.append("For children the dose depends on age and weight - please confirm with a pharmacist or pediatrician before giving any medicine.")
    L.append("(This is not a substitute for medical advice.)")
    return "\n".join(L)

def format_recommendation(info, shown, pool):
    keys, child = info["symptoms"], info["child"]
    if info["self_harm"]:
        return ("I'm really sorry you're going through this - you deserve real support right now. Please reach out to someone you trust "
                "or contact local emergency services (in Egypt: ambulance 123) or go to the nearest emergency room. I'm here to talk if you want to.")
    if info["red_flag"]:
        return ("⚠️ The symptoms you describe may be serious. Please go to the nearest emergency department or call emergency services "
                "right now, and do not self-medicate.")
    names = _symptom_names(keys)
    if not shown:
        return f"I couldn't find a suitable drug for ({names})" + (" for a child" if child else "") + " in my files. Please ask a pharmacist or doctor."
    if info["rank"]:
        return format_rank(info, shown, pool, from_history=False)

    L = [f"Get well soon! For ({names})" + (" in a child" if child else "") + ", these are the best matches in the medical guide and the pharmacy inventory:"]
    n = 0
    for title, source in (("From the medical guide:", "guide"), ("From the pharmacy inventory:", "inventory")):
        part = [x for x in shown if x["source"] == source]
        if part:
            L.append("\n" + title)
            for it in part:
                n += 1
                L.append(_fmt_item(n, it))
    if any(SYMPTOMS[k].get("rx") for k in keys):
        L.append("\nNote: these are prescription medicines - a doctor must decide the right treatment.")
    if child:
        L.append("\nFor children the dose depends on age and weight - please confirm with a pharmacist or pediatrician before giving any medicine.")
    L.append("\nIf symptoms get worse or last more than 3 days, see a doctor. (This is not a substitute for medical advice.)")
    return "\n".join(L)

# ---------- 8) Chat memory ----------
class ChatSession:
    """Keeps the chat history between turns and feeds it back into the workflow.
    - last_question / last_answer : for follow-ups like "what is the price of it?" and "summarize that"
    - pool  : every drug of the last recommendation (what "get the cheapest" compares)
    - shown : the drugs displayed last time (what "summarize" shortens to 2)
    """
    def __init__(self, app, vector_store):
        self.app, self.vector_store = app, vector_store
        self.last_question = self.last_answer = ""
        self.pool, self.shown = [], []

    def reset(self):
        self.last_question = self.last_answer = ""
        self.pool, self.shown = [], []

    def ask(self, task):
        state = {"task": task, "vector_store": self.vector_store, "retrieved_data": "", "processed_summary": "",
                 "final_output_path": "", "log": [],
                 "last_question": self.last_question, "last_answer": self.last_answer,
                 "last_pool": self.pool, "last_shown": self.shown}
        result = self.app.invoke(state)
        mode, answer = result.get("mode"), result["processed_summary"]
        if mode == "recommend":                       # a new list replaces the old one
            self.pool, self.shown = result.get("pool", []), result.get("shown", [])
            self.last_question, self.last_answer = task, answer
        elif mode == "rank":                          # same candidates, the displayed answer is the picked drug
            self.shown = result.get("shown", [])
            self.last_answer = answer
        elif mode in ("summarize_list", "summarize"):
            self.last_answer = answer
        elif mode in ("qa", "no_info"):               # a plain answer: the old list is no longer "the last answer"
            self.pool, self.shown = [], []
            self.last_question, self.last_answer = task, answer
        # smalltalk / nothing / nothing_to_rank: history stays as it was
        self.result = result
        return answer


In [ ]:
# ==============================================================================
# Step 4: Workflow Orchestration (Building the Graph)
# ==============================================================================
def build_workflow():
    print("Building Multi-Agent Workflow Graph...")
    workflow = StateGraph(AgentState)

    # Add Nodes (Agents)
    workflow.add_node("retriever", data_retrieval_agent)
    workflow.add_node("processor", processing_agent)
    workflow.add_node("actor", action_agent)

    # Define the sequential edges (Orchestration)
    workflow.set_entry_point("retriever")
    workflow.add_edge("retriever", "processor")
    workflow.add_edge("processor", "actor")
    workflow.add_edge("actor", END)

    # Compile the graph
    app = workflow.compile()
    return app

In [ ]:
# ==============================================================================
# Step 5: Testing & Evaluation
# ==============================================================================
def run_automated_tests():
    print("=" * 50)
    print("🚀 Starting Automated Multi-Agent System Evaluation")
    print("=" * 50)

    pdf_path = "Comprehensive_Medical___Pharmacy_Guide__120_Drugs_.pdf"
    csv_path = "Pharmacy_Inventory_120_New_Drugs.csv"

    vector_store = load_and_process_documents(pdf_path, csv_path)
    app = build_workflow()

    # (question, text that MUST appear if the data was really read)
    test_cases = [
        ("What is the standard dosage of Zyrtec?",                    "10mg"),                  # PDF
        ("What is the maximum daily dose of Panadol Advance?",        "4000"),                  # PDF
        ("Which drug is used for acid reflux?",                       "Omeprazole"),            # PDF
        ("What is the stock quantity of Feldene in the inventory?",   "Stock Quantity: 12"),    # CSV
        ("Which shelf is Paramol on?",                                "Shelf A6"),              # CSV
        ("What does the FDA label say about ibuprofen warnings?",     "openFDA"),               # API
        # the user describes a problem in his own words -> drug recommendation
        ("I have a cold, recommend me a medicine",                    "Congestal Tabs"),        # symptom -> PDF
        ("I have a cold, recommend me a medicine",                    "123 Cold"),              # symptom -> CSV inventory
        ("I have a fever",                                            "Paracetamol"),           # symptom
        ("My child has a fever",                                      "Paramol Syrup"),         # child -> only child-safe drugs
        ("I have heartburn",                                          "Omez"),                  # symptom
    ]

    retrieval_ok = answer_ok = 0
    for i, case in enumerate(test_cases):
        test_task, expected = case[0], case[1]
        print(f"\n--- Test Case {i+1} ---")
        print(f"Task: '{test_task}'")

        initial_state = {
            "task": test_task,
            "vector_store": vector_store,
            "retrieved_data": "",
            "processed_summary": "",
            "final_output_path": "",
            "log": []
        }

        result_state = app.invoke(initial_state)

        # Check 1: did the retrieval agent pull the right facts from the files? (data is read correctly)
        r_ok = expected.lower() in result_state["retrieved_data"].lower()
        # Check 2: did the final answer contain it too? (TinyLlama is small, so this can fail even if retrieval is fine)
        a_ok = expected.lower() in result_state["processed_summary"].lower()
        retrieval_ok += r_ok; answer_ok += a_ok

        print("\n[Final Output]")
        print(result_state["processed_summary"])
        print(f"-> Retrieval found '{expected}': {'PASS' if r_ok else 'FAIL'}")
        print(f"-> Answer contains '{expected}': {'PASS' if a_ok else 'FAIL'}")
        print(f"-> Report saved at {result_state['final_output_path']}")

    n = len(test_cases)
    print("\n" + "=" * 50)
    print(f"Retrieval accuracy (data read correctly): {retrieval_ok}/{n}")
    print(f"LLM answer accuracy                     : {answer_ok}/{n}")
    return vector_store


def run_conversation_tests(vector_store):
    # Multi-turn test: the system must understand the chat history.
    print("\n" + "=" * 50)
    print("💬 Conversation (chat history) tests")
    print("=" * 50)
    chat = ChatSession(build_workflow(), vector_store)
    checks = []

    def turn(msg):
        print(f"\nUser: {msg}")
        out = chat.ask(msg)
        print(f"Assistant:\n{out}")
        return out

    a1 = turn("I have a cold, recommend me a medicine")
    checks.append(("recommendation lists drugs", "Congestal Tabs" in a1))

    a2 = turn("summarize")
    n_drugs = len(re.findall(r"^\d+\. ", a2, flags=re.M))
    checks.append(("'summarize' keeps exactly 2 drugs", n_drugs == 2))

    cheapest = pick_by_price(chat.pool, "cheapest")["name"]
    priciest = pick_by_price(chat.pool, "priciest")["name"]
    a3 = turn("get the cheapest")
    checks.append((f"'get the cheapest' picks {cheapest} from the previous list", cheapest in a3 and "cheapest" in a3))

    a4 = turn("and the most expensive one?")
    checks.append((f"'most expensive' picks {priciest} from the previous list", priciest in a4 and "most expensive" in a4))

    a5 = turn("I have a headache, give me the cheapest medicine")
    checks.append(("symptom + cheapest in one message", "headache" in a5.lower() and "cheapest" in a5))

    chat.reset()
    a6 = turn("get the cheapest")
    checks.append(("'cheapest' with no history asks for a symptom first", a6 == NOTHING_TO_RANK))

    print("\n" + "-" * 50)
    for name, ok in checks:
        print(f"{'PASS' if ok else 'FAIL'}  {name}")
    print(f"Conversation tests: {sum(ok for _, ok in checks)}/{len(checks)}")


if __name__ == "__main__":
    vs = run_automated_tests()
    run_conversation_tests(vs)
